In [1]:
!git clone https://github.com/nickkats1/Prompting

Cloning into 'Prompting'...
remote: Enumerating objects: 169, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 169 (delta 57), reused 139 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (169/169), 1.77 MiB | 30.27 MiB/s, done.
Resolving deltas: 100% (57/57), done.


# LLM From Scratch

I will be using the GPT2 varient of the transformer. Most modern LLM's do not use this. I am not going to show full seq-to-seq due to the inconvenience

In [1]:
import torch
import torch.nn as nn
import tiktoken
from pathlib import Path
import sys
sys.path.append("/content/Prompting")

data_path = Path("/content/training_data.txt")

text_data = data_path.read_text(encoding="utf-8")
tokenizer = tiktoken.get_encoding("gpt2")

print(f"Training data loaded from {data_path}")

Training data loaded from /content/training_data.txt


In [2]:
!git clone https://github.com/nickkats1/Prompting

fatal: destination path 'Prompting' already exists and is not an empty directory.


In [3]:
total_chars = len(text_data)
print("total characters: ")
print(total_chars)
print("total tokens")
print(len(tokenizer.encode(text_data)))

total characters: 
18632532
total tokens
5536537


## Splitting the data

This has 5536537 tokens. Splitting the data is crucial for evaluating the model. This does not have to be said

In [4]:
train_ratio = 0.80
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

In [5]:
print("length of training data")
print(len(train_data))
print()
print("length of val data")
print()
print(len(val_data))

length of training data
14906025

length of val data

3726507


### Put data into GPTDataset and then put that data into dataloader

In [6]:
from transformer.dataset import create_dataloader
from transformer.config import CONFIG

train_loader = create_dataloader(
    train_data,
    batch_size=4,
    max_length=CONFIG["context_length"],
    stride=CONFIG["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader(
    val_data,
    batch_size=4,
    max_length=CONFIG["context_length"],
    stride=CONFIG["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [7]:
next(iter(train_loader))

[tensor([[   13,  9676,    11,  ...,   355, 48177, 29512],
         [  198,   198,    69,  ...,   286,   262,  1989],
         [ 7419,   529,  1146,  ...,  1690,   973,   287],
         [  981,    11,   910,  ...,   198, 30950,    12]]),
 tensor([[ 9676,    11,   428,  ..., 48177, 29512,    11],
         [  198,    69,    13,  ...,   262,  1989,  1022],
         [  529,  1146,  2742,  ...,   973,   287,  1295],
         [   11,   910,   351,  ..., 30950,    12, 16694]])]

In [8]:
next(iter(val_loader))

[tensor([[   16,   796,   513,  ...,   262,  6617,   286],
         [  428, 44728,   355,  ...,  1851,   856, 38226],
         [   13,   198, 36621,  ...,  1267,    32, 14095],
         [   16,   796,   317,  ...,    11,   597,  3716]]),
 tensor([[  796,   513,    87,  ...,  6617,   286,   428],
         [44728,   355,   281,  ...,   856, 38226,    13],
         [  198, 36621,    13,  ...,    32, 14095,    16],
         [  796,   317,  3539,  ...,   597,  3716,   198]])]

In [9]:
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

Train loader:
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 1024]) torch.Size([4, 1024])
torch.Size([4, 102

In [10]:
from torch.cuda import is_available
from transformer.utils import calc_loss_batch, calc_loss_loader
from transformer.model import GPTModel
device = ("cuda:0" if torch.cuda.is_available() else "cpu")
model = GPTModel(cfg=CONFIG)
model.to(device)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (attn): SelfAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (attn): SelfAttention(
        (W_query): Linear(in_features=768, out_featur

In [11]:
from transformer.trainer import train
torch.manual_seed(123)

model = GPTModel(CONFIG)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

num_epochs = 10
train_losses, val_losses, tokens_seen = train(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Winston Reed", tokenizer=tokenizer
)

Ep 1 (Step 000000): Train loss 9.538, Val loss 9.676
Winston Reed                                                  
Ep 1 (Step 000001): Train loss 9.538, Val loss 9.676
Winston Reed                                                  
Ep 1 (Step 000002): Train loss 9.538, Val loss 9.676
Winston Reed..................................................
Ep 1 (Step 000003): Train loss 9.538, Val loss 9.676
Winston Reed..................................................
Ep 1 (Step 000004): Train loss 9.538, Val loss 9.676
Winston Reed..................................................
Ep 1 (Step 000005): Train loss 8.210, Val loss 8.151
Winston Reed                                                  
Ep 1 (Step 000006): Train loss 8.210, Val loss 8.151
Winston Reed                                                  
Ep 1 (Step 000007): Train loss 8.210, Val loss 8.151
Winston Reed                                                  
Ep 1 (Step 000008): Train loss 8.210, Val loss 8.151
Winston Reed       

In [12]:
vocab = {
    "Winston Reed": 0,
    "Sterlin": 1,
    "Brutes": 2,
    "brutes": 3,
}

inverse_vocab = {v: k for k, v in vocab.items()}


next_token_logits = torch.tensor(
    [4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)

probas = torch.softmax(next_token_logits, dim=0)
next_token_id = torch.argmax(probas).item()

print(inverse_vocab[next_token_id])

brutes


In [24]:

torch.manual_seed(123)
next_token_id = torch.multinomial(probas, num_samples=1).item()
print(inverse_vocab[next_token_id])

brutes


In [26]:
top_k = 3
top_logits, top_pos = torch.topk(next_token_logits, top_k)

print("Top logits:", top_logits)
print("Top positions:", top_pos)

Top logits: tensor([6.7500, 6.2800, 4.5100])
Top positions: tensor([3, 7, 0])


### Generating samples

In [46]:
from transformer.utils import generate, token_ids_to_text, text_to_token_ids
from transformer.config import CONFIG

inference_device = torch.device("cuda")

torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Winston Reed", tokenizer).to(inference_device),
    max_new_tokens=35,
    context_size=CONFIG["context_length"],
    top_k=100,
    temperature=1.1
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Winston Reed.

The idea—3π
5.4.9.1;

is the arearibe
In different at free at2
10. a plane


In [47]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    },
    "model_and_optimizer.pth"
)